In [1]:
import sys
print(sys.executable)

!which python

/Users/efavila/Documents/Code/.venv/bin/python
/Users/efavila/Documents/Code/.venv/bin/python


In [2]:
from oilpriceapi import OilPriceAPI
import pandas as pd
from datetime import datetime, timedelta

In [3]:
# Inicializa el cliente (necesitas tu API key)
# Si la tienes en variable de entorno OILPRICEAPI_KEY, funciona solo
client = OilPriceAPI(api_key="a00c5bf30703cba90a06c9773bde691c47c8799a1d7351869a08988da284589b")

In [7]:
# 1. Obtener precios actuales de WTI y Brent
prices = client.prices.get_multiple(["WTI_USD", "BRENT_USD"])
for price in prices:
    print(f"{price.commodity}: ${price.value:.2f} a las {price.timestamp}")

# 2. Obtener datos históricos (para tu dataset)
end_date = datetime.now()
start_date = end_date - timedelta(days=30)  # Últimos 30 días

df = client.prices.to_dataframe(
    commodity="WTI_USD",
    start=start_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    interval="hourly"  # Puede ser "daily", "hourly", "minute"
)

print(f"Descargados {len(df)} registros")
print(df.head())

df_minute = client.prices.to_dataframe(
    commodity="WTI_USD",
    start="2026-03-02",
    end="2026-03-03",
    interval="minute"
)

print(f"Registros: {len(df_minute)}")
print(df_minute.head(10))
print(df_minute.columns.tolist())
print(df_minute.dtypes)

# 3. Ver qué más hay disponible
commodities = client.commodities.list()


WTI_USD: $73.52 a las 2026-03-03 08:30:38.832000+00:00
BRENT_CRUDE_USD: $80.56 a las 2026-03-03 08:40:26.853000+00:00
Descargados 500 registros
                          commodity  value currency    unit       type_name
date                                                                       
2026-02-10 13:00:00+00:00   WTI_USD  64.54      USD  barrel  hourly_average
2026-02-10 14:00:00+00:00   WTI_USD  64.47      USD  barrel  hourly_average
2026-02-10 15:00:00+00:00   WTI_USD  64.43      USD  barrel  hourly_average
2026-02-10 16:00:00+00:00   WTI_USD  64.05      USD  barrel  hourly_average
2026-02-10 17:00:00+00:00   WTI_USD  63.98      USD  barrel  hourly_average
Registros: 100
                                 commodity  value currency    unit   type_name
date                                                                          
2026-03-02 08:42:50.390000+00:00   WTI_USD  72.42      USD  barrel  spot_price
2026-03-02 08:50:34.883000+00:00   WTI_USD  72.27      USD  barrel  spot

In [10]:
"""BY DAY"""
df_day = client.prices.to_dataframe(
    commodity="WTI_USD",
    start="2026-03-02",
    end="2026-03-03",
    interval="minute"
)
print(f"Total ticks: {len(df_day)}")
print(f"Horas cubiertas: {df_day.index.min()} a {df_day.index.max()}")

# ¿Hay gaps grandes?
df_day['gap_minutes'] = df_day.index.to_series().diff().dt.total_seconds() / 60
print(f"\nGap promedio: {df_day['gap_minutes'].mean():.1f} min")
print(f"Gap máximo: {df_day['gap_minutes'].max():.1f} min")

Total ticks: 100
Horas cubiertas: 2026-03-02 08:42:50.390000+00:00 a 2026-03-02 13:30:58.624000+00:00

Gap promedio: 2.9 min
Gap máximo: 10.2 min


In [11]:
"""Identify gap"""
# Probar si el límite es por requests o por registros
df_test1 = client.prices.to_dataframe(
    commodity="WTI_USD",
    start="2026-03-01",  # Un día anterior completo
    end="2026-03-02",
    interval="minute"
)
print(f"Día anterior - Total ticks: {len(df_test1)}")
print(f"Horas: {df_test1.index.min()} a {df_test1.index.max()}")

# Probar con hourly para ver cobertura real
df_hourly = client.prices.to_dataframe(
    commodity="WTI_USD", 
    start="2026-02-01",
    end="2026-03-03",
    interval="hourly"
)
print(f"\nDatos hourly 30 días: {len(df_hourly)} registros")
print(f"Horas únicas por día promedio: {len(df_hourly) / 30:.1f}")

Día anterior - Total ticks: 100
Horas: 2026-03-02 08:50:34.883000+00:00 a 2026-03-02 13:38:37.927000+00:00

Datos hourly 30 días: 500 registros
Horas únicas por día promedio: 16.7


## Trying Yahoo Finance

In [12]:
import yfinance as yf
import pandas as pd

# Datos diarios - cobertura larga
cl_daily = yf.download("CL=F", start="2020-01-01", end="2026-03-03", interval="1d")
print("DIARIO:")
print(f"Registros: {len(cl_daily)}")
print(cl_daily.tail())

# Datos horarios - balance entre granularidad y cobertura
cl_hourly = yf.download("CL=F", period="2y", interval="1h")
print("\nHORARIO (2 años):")
print(f"Registros: {len(cl_hourly)}")
print(cl_hourly.tail())

# Verificar columnas
print("\nColumnas disponibles:", cl_daily.columns.tolist())

[*********************100%***********************]  1 of 1 completed


DIARIO:
Registros: 1550
Price           Close       High        Low       Open  Volume
Ticker           CL=F       CL=F       CL=F       CL=F    CL=F
Date                                                          
2026-02-24  65.629997  67.150002  65.550003  66.309998  312124
2026-02-25  65.419998  66.599998  65.120003  66.070000  306780
2026-02-26  65.209999  66.709999  63.599998  65.650002  498542
2026-02-27  67.019997  67.830002  64.849998  65.349998  437053
2026-03-02  71.230003  75.330002  69.199997  75.000000  437053


[*********************100%***********************]  1 of 1 completed


HORARIO (2 años):
Registros: 11205
Price                          Close       High        Low       Open Volume
Ticker                          CL=F       CL=F       CL=F       CL=F   CL=F
Datetime                                                                    
2026-03-03 04:00:00+00:00  72.669998  72.790001  72.269997  72.449997   4318
2026-03-03 05:00:00+00:00  72.860001  73.379997  72.540001  72.680000  11901
2026-03-03 06:00:00+00:00  73.059998  73.099998  72.330002  72.860001   9644
2026-03-03 07:00:00+00:00  73.839996  74.110001  73.010002  73.070000  23647
2026-03-03 08:00:00+00:00  74.150002  74.190002  73.480003  73.860001  20991

Columnas disponibles: [('Close', 'CL=F'), ('High', 'CL=F'), ('Low', 'CL=F'), ('Open', 'CL=F'), ('Volume', 'CL=F')]
